# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get("PurpleElegantBass749671")
print(f"Token loaded: {hf_token[:6]}... (length {len(hf_token)})" if hf_token else "Token is empty/None!")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
DECISION_MOMENT = "2026-04-30"
WINDOW_START = "2026-01-30"
SPLIT_DATE = "2026-03-15"

#read the daily content performance data from Parquet files covering January through April 2026
#filter it to the specific feature window between WINDOW_START and DECISION_MOMENT
#group the data by client and page
#split impressions into an early period (imp_early) and a later period (imp_late) based on SPLIT_DATE
#calculates total impressions, total clicks, and average search position for the entire window
#produce a DataFrame with one row per client-page combination

feature_paths = [f"{rel}/fact_content_daily_performance/month=2026-0{m}/*.parquet" for m in [1, 2, 3, 4]]

feature_df = con.sql(f"""
    WITH daily AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet([{', '.join(f"'{p}'" for p in feature_paths)}])
        WHERE report_date BETWEEN DATE '{WINDOW_START}' AND DATE '{DECISION_MOMENT}'
    )


    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_early,
        SUM(CASE WHEN report_date >  DATE '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_late,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,
        AVG(gsc_avg_position) AS avg_position_90d
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

qualifying = con.sql(f"""
    SELECT c.content_hash_id, c.client_hash_id, c.content_type, c.main_intent,
           c.word_count, c.char_count, c.competition_level, c.search_volume
    FROM read_parquet('{rel}/dim_content.parquet') c
    JOIN read_parquet('{rel}/dim_clients.parquet') cl USING (client_hash_id)
    WHERE c.content_created_date <= DATE '{WINDOW_START}'
      AND cl.gsc_data_start <= DATE '{WINDOW_START}'
""").df()

feature_df = feature_df.merge(qualifying, on=["client_hash_id", "content_hash_id"], how="inner")
feature_df["ctr"] = np.where(
    feature_df["impressions_90d"] > 0,
    (feature_df["clicks_90d"] / feature_df["impressions_90d"]) * 100,
    np.nan,
)
feature_df["is_declining"] = (feature_df["imp_late"] < feature_df["imp_early"]).astype(int)

print(f"Rows: {len(feature_df):,}")
feature_df.head()

Token loaded: hf_IcB... (length 37)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 260,285


,client_hash_id,content_hash_id,imp_early,imp_late,impressions_90d,clicks_90d,avg_position_90d,content_type,main_intent,word_count,char_count,competition_level,search_volume,ctr,is_declining
0,client_e547b89c05043229,content_7292af5ee9096ce1,1438.0,1802.0,3240.0,2.0,35.669767,keyword article,informational,1471,9089,LOW,50,0.061728,0
1,client_e547b89c05043229,content_14a6ade39e9a8e31,824.0,1770.0,2594.0,3.0,21.947559,keyword article,transactional,2996,18548,HIGH,20,0.115652,0
2,client_e547b89c05043229,content_5ed859ae9dc2e356,0.0,0.0,0.0,0.0,NaN,keyword article,transactional,1611,9874,LOW,10,NaN,0
3,client_e547b89c05043229,content_2618be372e19730f,400.0,709.0,1109.0,0.0,28.642764,keyword article,commercial,1497,9416,MEDIUM,10,0.000000,0
4,client_e547b89c05043229,content_1da7d268111c5875,1209.0,875.0,2084.0,3.0,19.449698,keyword article,transactional,2760,17157,LOW,10,0.143954,1


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(feature_df["client_hash_id"].nunique())
print(feature_df.groupby("client_hash_id").size().describe())

41
count       41.000000
mean      6348.414634
std       8415.829074
min         11.000000
25%        727.000000
50%       3026.000000
75%       9190.000000
max      31887.000000
dtype: float64


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**The target label — and why it isn't `is_declining`.** `w04`'s baseline rule *uses* `is_declining` (`imp_early` vs `imp_late`) as one of its inputs. If this model tried to predict that same flag, it would be circular — `is_declining` is a deterministic function of two features already sitting in `feature_df` (`imp_early`, `imp_late`), so a model "predicting" it would just be re-deriving arithmetic, not learning anything.

What's genuinely forward-looking instead: `w03`'s leakage-hunt check-only label — **did the page's impression rate keep falling into May, relative to the late-window rate?** (`still_declining_may`). This is the same label `w04`'s Signal 1 was validated against (verdict: MIXED), so scoring the model against it keeps the whole chain — baseline check, model target — consistent, and it's a genuinely non-circular ranking question: *"using only what's knowable by 2026-04-30, which pages are still declining into May?"* — exactly the "which first?" shape `training-honest-models` says needs precision@K.

Models, readable → stronger, matching the menu:
- **Logistic Regression** — the readable starting point.
- **Decision Tree** (`max_depth=4`) — shallow enough to read whole.
- **Random Forest** — the "does complexity earn its keep" check.

**Features used:** the observed 90-day window signals (`imp_early`, `imp_late`, `impressions_90d`, `clicks_90d`, `avg_position_90d`, `ctr`) plus `dim_content` metadata (`content_type`, `main_intent`, `word_count`, `char_count`, `competition_level`, `search_volume`) — all knowable by the decision moment. `is_declining` is deliberately **excluded** as a feature: it's a deterministic function of `imp_early`/`imp_late`, which are already included, so keeping it too would just double-count the same information under a different name. `client_hash_id` is used for grouping only, never as a feature (same convention as every prior notebook). Any May-derived column (`impressions_may`, `rate_may`) is the label source and never a feature.

In [5]:
may_df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_may
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-05/*.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

model_df = feature_df.merge(may_df, on=["client_hash_id", "content_hash_id"], how="left")
model_df["impressions_may"] = model_df["impressions_may"].fillna(0)

LATE_DAYS, MAY_DAYS = 46, 31
model_df["rate_late"] = model_df["imp_late"] / LATE_DAYS
model_df["rate_may"] = model_df["impressions_may"] / MAY_DAYS
model_df["still_declining_may"] = (model_df["rate_may"] < 0.8 * model_df["rate_late"]).astype(int)

print(f"Rows: {len(model_df):,}")
print(f"still_declining_may base rate: {model_df['still_declining_may'].mean():.3f}")

numeric_features = ["imp_early", "imp_late", "impressions_90d", "clicks_90d", "avg_position_90d", "ctr",
                     "word_count", "char_count", "search_volume"]
categorical_features = ["content_type", "main_intent", "competition_level"]
banned = {"is_declining", "impressions_may", "rate_late", "rate_may", "still_declining_may",
          "content_hash_id", "client_hash_id"}
assert not (set(numeric_features) | set(categorical_features)) & banned, "leakage/grouping column in feature list"
print(f"Numeric features: {len(numeric_features)}  |  Categorical features: {len(categorical_features)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 260,285
still_declining_may base rate: 0.289
Numeric features: 9  |  Categorical features: 3



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

`client_hash_id` is grouping-only, same reasoning as `w03`/`w04` — never a feature, always the boundary for train/test so no model can learn "this client's pages tend to decline" and get credit for memorizing accounts.

**But this population is more skewed than the CSV's 32 clients.** 41 clients, mean 6,348 rows each, **std 8,415** (bigger than the mean) — min 11 rows, max 31,887. A plain `GroupKFold(5)` with random client-to-fold assignment (what I used for the CSV rebuild) risks the same problem a single 80/20 split would have there: with only ~8 clients per fold, it's easy for one fold to catch two or three of the largest clients by chance while another fold gets only small ones — fold sizes and their reliability would swing wildly for reasons that have nothing to do with the model.

**Fix: size-balance the fold assignment.** Sort clients by row count descending, then greedily assign each client to whichever of the 5 folds currently has the smallest running total — a simple bin-packing approach. This keeps GroupKFold's core promise (a client's rows are never split across train and test) while also keeping the *folds themselves* roughly equal in size, so no fold's score is dominated by whichever giant client randomly landed there. Still out-of-fold predictions across all rows, same as before — just a fairer way to build the folds. Seed fixed at 42 throughout for the tie-breaking order.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
RANDOM_STATE = 42
N_FOLDS = 5

client_sizes = model_df.groupby("client_hash_id").size().sort_values(ascending=False)
fold_totals = np.zeros(N_FOLDS, dtype=int)
client_to_fold = {}
for client, size in client_sizes.items():
    smallest_fold = int(np.argmin(fold_totals))
    client_to_fold[client] = smallest_fold
    fold_totals[smallest_fold] += size

print("Rows per fold after size-balancing:")
for f in range(N_FOLDS):
    n_clients = sum(1 for v in client_to_fold.values() if v == f)
    print(f"  fold {f}: {fold_totals[f]:>7,} rows across {n_clients} clients")

model_df["fold"] = model_df["client_hash_id"].map(client_to_fold)

Rows per fold after size-balancing:
  fold 0:  52,005 rows across 7 clients
  fold 1:  52,065 rows across 9 clients
  fold 2:  52,165 rows across 7 clients
  fold 3:  51,998 rows across 9 clients
  fold 4:  52,052 rows across 9 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Metric: **precision@K** at K = 20, 50, 100, 200, plus the base rate as the "guessing randomly" floor — same convention as `w05`'s CSV version. The baseline's score is `w04`'s exact rule (`visible & position_ok & ctr_low & declining → impressions_90d`, else 0), recomputed on `model_df` and ranked against `still_declining_may` — the same label `w04`'s Signal 1 was already checked against, so this is a fair continuation, not a new comparison invented for this notebook.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

def build_matrix(frame, columns=None):
    num = frame[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = frame[categorical_features].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
    mat = pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)
    if columns is not None:
        mat = mat.reindex(columns=columns, fill_value=0)
    return mat

y = model_df["still_declining_may"].to_numpy()

model_builders = {
    "logistic_regression": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "decision_tree": lambda: DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=200, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "random_forest": lambda: RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=50,
        class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
    ),
}

oof_scores = {name: np.zeros(len(model_df)) for name in model_builders}
for f in range(N_FOLDS):
    train_mask = model_df["fold"] != f
    test_mask = model_df["fold"] == f
    X_train = build_matrix(model_df[train_mask])
    X_test = build_matrix(model_df[test_mask], columns=X_train.columns)
    y_train = y[train_mask.to_numpy()]
    for name, builder in model_builders.items():
        model = builder()
        model.fit(X_train, y_train)
        oof_scores[name][test_mask.to_numpy()] = model.predict_proba(X_test)[:, 1]
    print(f"fold {f}: train rows={train_mask.sum():,}  test rows={test_mask.sum():,}")

# baseline score, w04's exact rule, recomputed here
visible = model_df["impressions_90d"] >= 500
position_ok = (model_df["avg_position_90d"] > 0) & (model_df["avg_position_90d"] <= 20)
ctr_low = model_df["ctr"] < 0.4
declining = model_df["is_declining"] == 1
flag = visible & position_ok & ctr_low & declining
baseline_score = np.where(flag, model_df["impressions_90d"], 0)
print(f"\nBaseline flagged: {flag.sum():,} / {len(model_df):,}")

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

rows = []
for k in [20, 50, 100, 200]:
    row = {"k": k, "base_rate": round(y.mean(), 3), "baseline": round(precision_at_k(y, baseline_score, k), 3)}
    for name in model_builders:
        row[name] = round(precision_at_k(y, oof_scores[name], k), 3)
    rows.append(row)

comparison_table = pd.DataFrame(rows).set_index("k")
comparison_table



fold 0: train rows=208,280  test rows=52,005
fold 1: train rows=208,220  test rows=52,065
fold 2: train rows=208,120  test rows=52,165
fold 3: train rows=208,287  test rows=51,998
fold 4: train rows=208,233  test rows=52,052

Baseline flagged: 22,971 / 260,285


,base_rate,baseline,logistic_regression,decision_tree,random_forest
k,,,,,
20,0.289,0.60,0.30,0.900,0.90
50,0.289,0.62,0.42,0.940,0.82
100,0.289,0.61,0.43,0.890,0.82
200,0.289,0.62,0.41,0.845,0.84


**What the table says, in plain terms:**

Baseline beats base rate this time (0.60–0.62 vs 0.289) — the opposite of the CSV rebuild, where the baseline scored below base rate. Worth stating as a real contrast: this warehouse baseline's rule is actually predictive of still_declining_may, not just directionally weak.
Decision Tree wins almost every K — 0.90 / 0.94 / 0.89 / 0.845 — and Random Forest is close behind, tying DT exactly at K=20 (0.90) but trailing everywhere else.
Logistic Regression is the surprise failure — 0.30 / 0.42 / 0.43 / 0.41, worse than the baseline at every single K, and at K=20 (0.30) it's barely above the 0.289 base rate — close to guessing randomly. That's a real, honest finding, not something to bury: the relationship here likely isn't linear (a threshold/momentum effect a tree can split on cleanly, that a straight line can't capture), and it's worth saying plainly rather than picking a rosier K to lead with.

All three models beat the base rate, and the decision tree/random forest post strong
precision@K numbers (0.82-0.94) against a baseline that itself beats base rate this time
(0.60-0.62 vs 0.289 — unlike the CSV rebuild, this rule is genuinely predictive here).
Logistic regression is the clear failure, underperforming the baseline at every K and
barely clearing base rate at K=20 (0.30) — the relationship isn't linear.

But the decision tree's win needs a major caveat, covered in Section 4: permutation
importance shows it relies almost entirely on a single feature (imp_late), and the
mechanism is closer to an artifact of how the label was constructed than a real
content-quality signal.

All three models beat base rate, and the baseline itself beats base rate too (0.60-0.62 vs
0.289) — unlike the CSV rebuild, this rule is genuinely predictive of this label. Decision
tree and random forest post strong numbers (0.82-0.94), logistic regression is the clear
failure (0.30-0.43, worse than the baseline at every K) — the relationship isn't linear.

But the tree's apparent win needs a major caveat, covered fully in Section 4: it relies
almost entirely on one feature (imp_late), which turns out to be a mechanical artifact of
how the label is constructed, not a real content-quality signal.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Replace "random_forest" below with whichever model actually wins in your Section 3 table.
best_model_name = "decision_tree"
best_final = model_builders[best_model_name]()
X_full = build_matrix(model_df)
best_final.fit(X_full, y)

pi = permutation_importance(best_final, X_full, y, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
from sklearn.tree import export_text
print(export_text(best_final, feature_names=list(X_full.columns), max_depth=3))

importance = pd.Series(pi.importances_mean, index=X_full.columns).sort_values(ascending=False)
print(f"Top 10 permutation-importance features ({best_model_name}):")
print(importance.head(10).round(4))

df_err = model_df.copy()
df_err["oof_score"] = oof_scores[best_model_name]

print("\nClient concentration in the top 50 highest-scored rows:")
print(df_err.sort_values("oof_score", ascending=False).head(50)["client_hash_id"].value_counts().head(5))

print("\n3 concrete false positives (scored high, did not keep declining into May):")
fp = df_err[df_err["still_declining_may"] == 0].sort_values("oof_score", ascending=False).head(3)
print(fp[["content_hash_id", "client_hash_id", "oof_score", "imp_early", "imp_late", "impressions_may", "avg_position_90d", "ctr"]].to_string(index=False))

print("\n3 concrete false negatives (kept declining, scored lowest):")
fn = df_err[df_err["still_declining_may"] == 1].sort_values("oof_score").head(3)
print(fn[["content_hash_id", "client_hash_id", "oof_score", "imp_early", "imp_late", "impressions_may", "avg_position_90d", "ctr"]].to_string(index=False))

|--- imp_late <= 0.50
|   |--- class: 0
|--- imp_late >  0.50
|   |--- imp_late <= 1.50
|   |   |--- content_type_feedly article <= 0.50
|   |   |   |--- word_count <= 1256.50
|   |   |   |   |--- class: 1
|   |   |   |--- word_count >  1256.50
|   |   |   |   |--- class: 1
|   |   |--- content_type_feedly article >  0.50
|   |   |   |--- word_count <= 1156.00
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  1156.00
|   |   |   |   |--- class: 1
|   |--- imp_late >  1.50
|   |   |--- char_count <= 4593.50
|   |   |   |--- avg_position_90d <= 16.75
|   |   |   |   |--- class: 1
|   |   |   |--- avg_position_90d >  16.75
|   |   |   |   |--- class: 1
|   |   |--- char_count >  4593.50
|   |   |   |--- ctr <= 0.15
|   |   |   |   |--- class: 1
|   |   |   |--- ctr >  0.15
|   |   |   |   |--- class: 1

Top 10 permutation-importance features (decision_tree):
imp_late                       0.3041
content_type_feedly article    0.0070
word_count                     0.0038
imp_ear

**Run without imp_early and imp_late**

In [9]:
reduced_features = [f for f in numeric_features if f not in ("imp_early", "imp_late")]

def build_matrix_reduced(frame, columns=None):
    num = frame[reduced_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = frame[categorical_features].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
    mat = pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)
    if columns is not None:
        mat = mat.reindex(columns=columns, fill_value=0)
    return mat

oof_reduced = np.zeros(len(model_df))
for f in range(N_FOLDS):
    train_mask = model_df["fold"] != f
    test_mask = model_df["fold"] == f
    X_train = build_matrix_reduced(model_df[train_mask])
    X_test = build_matrix_reduced(model_df[test_mask], columns=X_train.columns)
    y_train = y[train_mask.to_numpy()]
    tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, class_weight="balanced", random_state=RANDOM_STATE)
    tree.fit(X_train, y_train)
    oof_reduced[test_mask.to_numpy()] = tree.predict_proba(X_test)[:, 1]

print("Precision@K WITHOUT imp_early/imp_late:")
for k in [20, 50, 100, 200]:
    print(f"  k={k}: {precision_at_k(y, oof_reduced, k):.3f}   (base_rate={y.mean():.3f}, full-feature tree was {precision_at_k(y, oof_scores['decision_tree'], k):.3f})")

X_full_reduced = build_matrix_reduced(model_df)
reduced_final = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, class_weight="balanced", random_state=RANDOM_STATE)
reduced_final.fit(X_full_reduced, y)
pi_reduced = permutation_importance(reduced_final, X_full_reduced, y, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp_reduced = pd.Series(pi_reduced.importances_mean, index=X_full_reduced.columns).sort_values(ascending=False)
print("\nTop 10 permutation-importance features (no imp_early/imp_late):")
print(imp_reduced.head(10).round(4))

print("\n" + export_text(reduced_final, feature_names=list(X_full_reduced.columns), max_depth=3))

Precision@K WITHOUT imp_early/imp_late:
  k=20: 0.650   (base_rate=0.289, full-feature tree was 0.900)
  k=50: 0.760   (base_rate=0.289, full-feature tree was 0.940)
  k=100: 0.700   (base_rate=0.289, full-feature tree was 0.890)
  k=200: 0.625   (base_rate=0.289, full-feature tree was 0.845)

Top 10 permutation-importance features (no imp_early/imp_late):
impressions_90d                    0.2433
content_type_feedly article        0.0082
main_intent_unknown                0.0058
main_intent_commercial             0.0000
clicks_90d                         0.0000
char_count                         0.0000
ctr                                0.0000
search_volume                      0.0000
content_type_comparison article    0.0000
content_type_keyword article       0.0000
dtype: float64

|--- impressions_90d <= 1.50
|   |--- impressions_90d <= 0.50
|   |   |--- main_intent_informational <= 0.50
|   |   |   |--- class: 0
|   |   |--- main_intent_informational >  0.50
|   |   |   |--- class:

**What the tree actually learned — and why the precision@K numbers overstate it.**
Permutation importance shows the full-feature tree depends almost entirely on `imp_late`
(0.304 of 0.309 total importance) — every content or ranking feature sits at ~0. The label,
`still_declining_may = rate_may < 0.8 * rate_late`, has a floor effect: when `imp_late` is
already near zero, there's almost no room left to drop another 20% in relative terms, so the
label lands on "still declining" almost by construction, not because of anything the page did.

To test this, I re-ran the tree WITHOUT imp_early/imp_late. Precision@K dropped
(0.90→0.65 at K=20, 0.94→0.76 at K=50) but stayed well above base rate — so it isn't nothing.
But permutation importance shows the tree just found the next-best proxy for the same floor
effect: `impressions_90d` alone now carries 0.243 importance, with splits at tiny absolute
thresholds (0.5, 1.5, 3.5, 11.5). `avg_position_90d` and `ctr` appear in the tree's branches
but carry ~0 permutation importance — not actually driving predictions.

**Honest conclusion:** this model is mostly learning "very low overall search volume," which
mechanically correlates with a relative-decline label, not "declining, worth a content
review." The high precision@K numbers are real but shouldn't be read as "the model
understands content decline" — a genuinely useful model would need a differently-built label
(e.g. an absolute impressions floor before computing rate_may/rate_late, so near-zero pages
can't trivially qualify) to separate real signal from this artifact. That's a good next
iteration, not something this notebook needed to solve today.

**False positives** (imp_early 7-25, imp_late 1-15, zero CTR, position 5.6-12.2) are small
counts that bounced back in May (88, 7, 5 impressions) — noise around a low count, not a
reversed trend.

**False negatives** (imp_early/imp_late both 0-1, impressions_may=0) had no real signal to
begin with — a cold-start gap.

**Client concentration.** Top 50 dominated by two clients (client_d211cb07b9059bab,
client_3197e6291363b4db — 23 rows each, 92% combined) — different accounts than w04's
baseline top 20, so the concentration problem persists but isn't the same specific clients
recurring.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.